# Web scraping clásico: BeautifulSoup

En esta sesión aprendemos los fundamentos:

1. Qué pasa cuando un navegador pide una página (HTTP)
2. Cómo está estructurado un HTML (DOM, etiquetas, clases)
3. Descargar páginas con `requests`
4. Extraer datos con `BeautifulSoup` (selectores CSS)
5. Paginación y construcción de un dataset con `pandas`
6. Pros y contras de este enfoque


## 0. Antes de scrapear: ética y legalidad

- Revisa el **`robots.txt`** del sitio.
- Lee los **términos de uso**. Que algo sea público no significa que se pueda reutilizar.
- **No satures el servidor**: añade pausas (`time.sleep`) entre peticiones.
- Cuidado con los **datos personales** (RGPD).
- Si existe una **API oficial**, úsala antes que scrapear.

In [ ]:
# pip install beautifulsoup4

In [1]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

## 1. Descargar una página con `requests`

Una petición HTTP `GET` devuelve:
- un **código de estado** (200 = OK, 404 = no existe, 403 = prohibido, 429 = demasiadas peticiones…)
- **cabeceras**
- el **cuerpo**: el HTML

In [2]:
BASE_URL = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0 (bootcamp-data-science; fines educativos)"}

response = requests.get(BASE_URL, headers=HEADERS, timeout=10)

print("Código de estado:", response.status_code)
print("Tipo de contenido:", response.headers.get("Content-Type"))
print("Longitud del HTML:", len(response.text), "caracteres")
print(response.text[:500])

Código de estado: 200
Tipo de contenido: text/html
Longitud del HTML: 51294 caracteres
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


## 2. Parsear el HTML con BeautifulSoup

`BeautifulSoup` convierte el texto HTML en un **árbol navegable**.

**Truco:** en el navegador, clic derecho/*Inspeccionar* sobre un libro. Verás que cada libro está dentro de un `<article class="product_pod">`.

In [3]:
soup = BeautifulSoup(response.content, "lxml")

print(soup.title.text)


    All products | Books to Scrape - Sandbox



### Dos formas de buscar elementos

| Método | Ejemplo | Devuelve |
|---|---|---|
| `find` / `find_all` | `soup.find_all("article", class_="product_pod")` | Primer elemento / lista |
| `select_one` / `select` (CSS) | `soup.select("article.product_pod")` | Primer elemento / lista |

Los **selectores CSS** son más potentes, y además nos servirán después en Playwright y Crawl4AI. Merece la pena dominarlos.

In [5]:
libros = soup.select("article.product_pod")
print(f"Libros en la página: {len(libros)}")
print(libros[0].prettify()[:1200])

Libros en la página: 20
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [6]:
primero = libros[0]

titulo = primero.select_one("h3 a")["title"]         
precio = primero.select_one("p.price_color").text      
stock  = primero.select_one("p.instock.availability").get_text(strip=True)
rating = primero.select_one("p.star-rating")["class"][1]
enlace = urljoin(BASE_URL, primero.select_one("h3 a")["href"])

print(titulo, precio, stock, rating, enlace, sep="\n")

A Light in the Attic
£51.77
In stock
Three
https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


## 3. Limpiar los datos

Los datos scrapeados casi nunca vienen listos para analizar: `'£51.77'` es texto y `'Three'` no es un número.

In [13]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_libro(article, base_url):
    a = article.select_one("h3 a")
    return {
        "titulo": a["title"],
        "precio_gbp": float(article.select_one("p.price_color").text.replace("£", "").replace("Â", "")),
        "rating": RATING_MAP.get(article.select_one("p.star-rating")["class"][1]),
        "en_stock": "In stock" in article.select_one("p.availability").text,
        "url": urljoin(base_url, a["href"]),
    }

pd.DataFrame([parse_libro(l, BASE_URL) for l in libros]).head()

,titulo,precio_gbp,rating,en_stock,url
0,A Light in the Attic,51.77,3,True,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,True,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,True,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,True,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,True,https://books.toscrape.com/catalogue/sapiens-a...


## 4. Paginación: recorrer todo el catálogo

Abajo del todo hay un botón **next** (`li.next a`). Lo seguimos hasta que desaparece.

In [ ]:
def scrape_catalogo(start_url, max_paginas=None, pausa=0.5):
    url, datos, pagina = start_url, [], 0
    while url:
        r = requests.get(url, headers=HEADERS, timeout=10)
        r.raise_for_status()
        s = BeautifulSoup(r.content, "lxml")

        datos += [parse_libro(a, url) for a in s.select("article.product_pod")]
        pagina += 1
        print(f"Página {pagina}: {len(datos)} libros")

        siguiente = s.select_one("li.next a")
        url = urljoin(url, siguiente["href"]) if siguiente else None

        if max_paginas and pagina >= max_paginas:
            break
        time.sleep(pausa)
    return pd.DataFrame(datos)

df = scrape_catalogo(BASE_URL, max_paginas=5)
df.shape

Página 1: 20 libros
Página 2: 40 libros
Página 3: 60 libros
Página 4: 80 libros
Página 5: 100 libros


(100, 5)

In [15]:
df.groupby("rating")["precio_gbp"].agg(["count", "mean"]).round(2)

,count,mean
rating,,
1,22,35.52
2,19,35.91
3,22,36.84
4,18,33.98
5,19,30.01


In [16]:
df.to_csv("data/libros.csv", index=False)

## 5. La limitación clave: páginas con JavaScript

Muchas webs modernas (React, Vue, Angular…) envían un HTML **casi vacío** y rellenan el contenido con JavaScript **en el navegador**. `requests` no ejecuta JavaScript, así que no ve esos datos.

Compara estas dos versiones de la misma web:

In [17]:
for url in ["https://quotes.toscrape.com/", "https://quotes.toscrape.com/js/"]:
    s = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=10).content, "lxml")
    print(f"{url:40} -> citas encontradas: {len(s.select('div.quote'))}")

https://quotes.toscrape.com/             -> citas encontradas: 10
https://quotes.toscrape.com/js/          -> citas encontradas: 0


La versión `/js/` devuelve **0 citas**: el HTML trae un `<script>` que genera las citas al cargarse en el navegador. Para eso necesitamos un **navegador real** → **Notebook 02: Playwright**.

## 6. Pros y contras de Requests + BeautifulSoup

| Pros | Contras |
|---|---|
| **Muy fácil de aprender**: ideal para entender HTTP y HTML | **No ejecuta JavaScript**: no sirve para SPAs/webs dinámicas |
| **Rapidísimo y ligero**: sin navegador, poca RAM y CPU | **Selectores frágiles**: si cambia una clase CSS, el scraper se rompe |
| Documentación enorme y muchísimos ejemplos | **Fácil de bloquear**: los sistemas anti-bot detectan peticiones "no humanas" |
| Control total de cada petición (cabeceras, cookies, sesiones) | Tú lo programas todo: reintentos, paginación, concurrencia, limpieza |
| Perfecto para webs estáticas, tablas y prototipos | No interactúa (clics, formularios, scroll infinito) |
| Sin dependencias pesadas, funciona en cualquier sitio | Escalar a miles de páginas exige más trabajo (async, colas…) |

**¿Cuándo usarlo?** Webs estáticas, prototipos rápidos, extracción puntual o como primer intento (si funciona, es lo más barato).